<a href="https://colab.research.google.com/github/venkatgkasturi/swot/blob/master/Exercise_File_e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Setting Up the Development Environment**
The notebook uses Microsoft GODEL (https://github.com/microsoft/GODEL), which is a knowledge-grounded dialog model available on the Hugging Face Hub.
Run the pip cell below to install `transformers`, `accelerate`, `safetensors`, and helpers that keep the notebook runnable in Colab or other Python 3.12 environments.

If you have a Hugging Face token, define `HUGGINGFACEHUB_API_TOKEN` (or `HF_TOKEN`) as an environment variable to download the model faster.


In [ ]:
!pip install "transformers>=4.36" accelerate safetensors torch gradio huggingface_hub


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'microsoft/GODEL-v1_1-base-seq2seq'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hf_token = os.environ.get('HUGGINGFACEHUB_API_TOKEN') or os.environ.get('HF_TOKEN')

print('Loading', MODEL_NAME, 'on device', device)
if hf_token:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_auth_token=hf_token)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, use_auth_token=hf_token)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()
print('Model warmed up with', sum(p.numel() for p in model.parameters()))


## **Building a Baseline Chatbot**
GODEL is trained with a format that concatenates the dialog context, an instruction, and an optional knowledge string. The cell below recreates that formatting and provides a callable `generate_response` helper.


In [ ]:
from typing import List, Tuple

instruction_template = 'Instruction: craft a helpful, grounded response for the user. '

dialog_history: List[Tuple[str, str]] = []
knowledge_base = [
    {
        'keyword': 'project',
        'text': 'Project schedule: the next sprint review is Tuesday at 10 a.m.'
    },
    {
        'keyword': 'benefits',
        'text': 'Employee benefits include monthly wellness stipends and reimbursement for certifications.'
    },
    {
        'keyword': 'support',
        'text': 'Support hours run from 8 a.m. to 6 p.m. Pacific, and we respond within one business day.'
    },
]

def select_knowledge(query: str) -> str:
    tokens = {token.strip(',.?').lower() for token in query.split()}
    best = max(knowledge_base, key=lambda entry: len(tokens & {entry['keyword']}), default=None)
    return best['text'] if best else ''

def format_dialog(dialog: List[Tuple[str, str]]) -> str:
    return ' EOS '.join([f"{role}: {text}" for role, text in dialog])

def generate_response(query: str, history: List[Tuple[str, str]]) -> Tuple[str, str]:
    knowledge = select_knowledge(query)
    parts = [f'Instruction: given the dialog context and knowledge, respond helpfully.']
    context = format_dialog(history)
    template = f"{' '.join(parts)} [CONTEXT] {context}"
    if knowledge:
        template += f" [KNOWLEDGE] {knowledge}"
    inputs = tokenizer(template, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_length=128, num_beams=4, do_sample=True, top_p=0.9)
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return reply, knowledge

print('Helper ready. Example response:')
example_history = [('user', 'Tell me how to prepare for a sprint review.'), ('assistant', 'Make sure to gather demos and be ready for questions.')]
print(generate_response('What should I bring up in the sprint review?', example_history))


## **Launch Your First Chatbot Locally**
You can wrap `generate_response` in a Gradio interface or a REST endpoint. The following cell demonstrates a minimal Gradio chat plus an example HTTP payload for any Flask/FastAPI wrapper.


In [ ]:
import gradio as gr

theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    neutral_hue="slate",
    radius_size="lg",
    text_size="md",
)

css = """
:root {
  --page-bg-1: #0b1020;
  --page-bg-2: #151b36;
  --card-bg: rgba(255, 255, 255, 0.08);
  --card-border: rgba(255, 255, 255, 0.14);
  --text-main: #f8fafc;
  --text-muted: #cbd5e1;
}

.gradio-container {
  background:
    radial-gradient(circle at top left, rgba(99,102,241,0.22), transparent 28%),
    radial-gradient(circle at top right, rgba(56,189,248,0.16), transparent 24%),
    linear-gradient(135deg, var(--page-bg-1), var(--page-bg-2));
  min-height: 100vh;
}

#app-shell {
  max-width: 980px;
  margin: 28px auto;
  border: 1px solid var(--card-border);
  border-radius: 24px;
  backdrop-filter: blur(14px);
  background: rgba(15, 23, 42, 0.58);
  box-shadow: 0 20px 60px rgba(0, 0, 0, 0.28);
  overflow: hidden;
}

#hero {
  padding: 28px 28px 8px 28px;
}

#hero h1 {
  margin: 0;
  font-size: 2rem;
  line-height: 1.1;
  color: var(--text-main);
}

#hero p {
  margin: 10px 0 0 0;
  color: var(--text-muted);
  font-size: 0.98rem;
}

#badge-row {
  display: flex;
  gap: 10px;
  flex-wrap: wrap;
  padding: 14px 28px 0 28px;
}

.badge {
  border: 1px solid var(--card-border);
  background: rgba(255,255,255,0.06);
  color: #e2e8f0;
  padding: 6px 10px;
  border-radius: 999px;
  font-size: 0.85rem;
}

#chat-wrap {
  padding: 18px 20px 24px 20px;
}

#footer-note {
  padding: 0 28px 24px 28px;
  color: var(--text-muted);
  font-size: 0.86rem;
}

/* Chat area */
[data-testid="chatbot"] {
  border-radius: 20px !important;
  border: 1px solid var(--card-border) !important;
  background: rgba(255,255,255,0.05) !important;
}

/* Input row */
footer {
  border-top: 0 !important;
}

textarea, .gr-textbox, .gr-textbox textarea {
  border-radius: 16px !important;
}

/* Buttons */
button.primary {
  border-radius: 14px !important;
}

/* Make markdown in messages cleaner */
.message, .message * {
  line-height: 1.5;
}
"""

def gradio_chat(message, history):
    history = history or []

    dialog = []
    for msg in history:
        dialog.append((msg["role"], msg["content"]))

    response, knowledge = generate_response(message, dialog)
    return response

with gr.Blocks(theme=theme, css=css) as demo:
    with gr.Column(elem_id="app-shell"):
        gr.HTML("""
        <div id="hero">
          <h1>GODEL Knowledge-Grounded Bot</h1>
          <p>
            Ask about support, benefits, or project updates and get answers grounded
            in your notebook knowledge snippets.
          </p>
        </div>
        <div id="badge-row">
          <span class="badge">Grounded answers</span>
          <span class="badge">Fast local demo</span>
          <span class="badge">Gradio chat UI</span>
        </div>
        """)

        with gr.Column(elem_id="chat-wrap"):
            gr.ChatInterface(
                fn=gradio_chat,
                type="messages",
                chatbot=gr.Chatbot(
                    height=540,
                    type="messages",
                    allow_tags=False,
                    placeholder="<strong>No messages yet.</strong><br/>Start by asking about support hours, benefits, or the sprint review."
                ),
                textbox=gr.Textbox(
                    placeholder="Ask a question...",
                    container=False,
                    scale=8,
                ),
                examples=[
                    "Tell me about the support hours",
                    "What benefits are available?",
                    "When is the next sprint review?"
                ],
                fill_height=True,
            )

        gr.HTML("""
        <div id="footer-note">
          Tip: this version uses Gradio's message-style chat format, which is the recommended path forward.
        </div>
        """)

print("Call demo.launch() to interact.")

## **Fine-Tuning the Chatbot for Better Conversations**
Fine-tuning GODEL is heavier, but you can still create a small downstream dataset GODEL expects: JSON lines with `Context`, `Knowledge`, and `Response` keys. The cell below writes a toy training file and explains how you might expand it.


In [ ]:
import json
from pathlib import Path

data = [
    {
        'Context': 'User: Can you remind me about the next sprint review? Assistant: Sure, what do you need?'
                   ' User: I want to highlight the new metrics dashboard.',
        'Knowledge': 'Sprint review is Tuesday at 10 a.m., and new metrics were added to Dashboard v2.',
        'Response': 'Mention the metrics dashboard update and the Tuesday review when you speak.'
    },
    {
        'Context': 'User: What is the best way to reach support if I discover a bug? Assistant: We usually reply within a day.'
                   ' User: Do they keep tickets in Slack?'
                   ,
        'Knowledge': 'Support hours: 8 a.m. to 6 p.m. Pacific. Submit tickets via support@company.com.',
        'Response': 'Send a note to support@company.com during Pacific business hours and we will respond within one day.'
    },
]
output = Path('godel_finetune.jsonl')
with output.open('w', encoding='utf8') as fh:
    for entry in data:
        fh.write(json.dumps(entry) + '\n')

print('Generated', output.absolute(), 'with', len(data), 'examples. Use it with the GODEL scripts in https://github.com/microsoft/GODEL to fine-tune the model.')


## **Further Upgrading Chatbot Responses**


### **Traceable Knowledge**
Log which knowledge snippet `generate_response` used so you can validate whether the bot cited the right fact. The helper already returns that knowledge string.


In [ ]:
chat_history = []
response, knowledge = generate_response('What are the support hours?', chat_history)
print('Knowledge used:', knowledge)
print('Generated reply:', response)


### **Context Awareness via History**
The `dialog_history` that you pass to `generate_response` is the context so GODEL can respect earlier turns. You can experiment with truncating or conditioning on particular spans below.


In [ ]:
long_history = [
    ('user', 'Tell me about the sprint review.'),
    ('assistant', 'You should show the new dashboard.'),
    ('user', 'And what about the metrics team?'),
]

print('Short context response:')
print(generate_response('Should I bring the metrics leader?', long_history))


### **Confidence-Based Fallbacks**
When no knowledge snippet matches, fall back to a clarification prompt instead of hallucinating. The helper below shows a simple fallback message.


In [ ]:
def ask_with_fallback(prompt: str, history: List[Tuple[str, str]]):
    answer, knowledge = generate_response(prompt, history)
    if not knowledge:
        return 'I am not sure I have enough context yet. Could you add more detail?'
    return answer

print(ask_with_fallback('What is the company motto?', []))


In [ ]:
demo.launch(debug=True)